# NeuroScan NG: VGG19 Training on Colab

Trains the Alzheimer's MRI classifier for the study *Design and Implementation of a Web-Based Intelligent System for Early Detection of Alzheimer's Disease from MRI Images using CNNs* (case study: Nigeria).

**Before running:** Runtime > Change runtime type > **T4 GPU**.

Steps: install deps, download the Kaggle 4-class dataset, upload `train_model.py` from the project zip, train, then download the `model/` artifacts and drop them into the web app.


In [ ]:
!pip -q install kagglehub matplotlib
import tensorflow as tf
print('TF', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))


## 1. Download and merge the dataset
The Kaggle dataset ships as `train/` and `test/` splits. We merge them into one folder of four class subfolders; `train_model.py` then makes its own stratified 70/15/15 split, which prevents the leakage issues flagged in the literature review.


In [ ]:
import kagglehub, os, shutil, glob
root = kagglehub.dataset_download('tourist55/alzheimers-dataset-4-class-of-images')
print('Downloaded to', root)

DATA = '/content/AlzheimerAll'
classes = ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']
for c in classes:
    os.makedirs(os.path.join(DATA, c), exist_ok=True)

count = 0
for split in ('train', 'test'):
    for c in classes:
        for src in glob.glob(os.path.join(root, '**', split, c, '*'), recursive=True):
            dst = os.path.join(DATA, c, f'{split}_{os.path.basename(src)}')
            shutil.copy(src, dst); count += 1
print('Merged images:', count)
for c in classes:
    print(f'  {c}:', len(os.listdir(os.path.join(DATA, c))))


## 2. Upload `train_model.py`
Pick `train_model.py` from the NeuroScan NG project folder when prompted.


In [ ]:
from google.colab import files
up = files.upload()
assert 'train_model.py' in up, 'Upload train_model.py from the project zip'


## 3. Train
Phase 1 trains the new head on the frozen VGG19 base; phase 2 fine-tunes block5 at a low learning rate. Expect roughly 30 to 60 minutes on a T4. All seven evaluation metrics are computed on the held-out test set.


In [ ]:
!python train_model.py --data_dir /content/AlzheimerAll --out_dir model --epochs 12 --fine_tune_epochs 6 --batch_size 32


## 4. Inspect the results


In [ ]:
import json
m = json.load(open('model/metrics.json'))
print(json.dumps({k: v for k, v in m.items() if k not in ('confusion_matrix', 'per_class')}, indent=2))
from IPython.display import Image as I, display
display(I('model/training_curves.png'))
display(I('model/confusion_matrix.png'))


## 5. Download the artifacts
Unzip into the web app's `model/` folder, `pip install tensorflow` where the app runs, restart, and the demo banner disappears: predictions come from this model, Grad-CAM heatmaps switch on, and the performance page shows these metrics.


In [ ]:
!cd model && zip -r ../neuroscan_model.zip .
files.download('neuroscan_model.zip')
